# 01 · Lecturer track: prediction from ICU hour 1, class balancing, KA-Transformer comparison

**What is predicted.** Every ICU hour, from hour 1 to the end of the stay, gets a prediction for the label at t+1, t+3 and t+6 h. Label definition unchanged: 1 iff sepsis onset is 0–6 h ahead; post-onset rows dropped. Evaluation is full coverage: every hour is scored exactly once.

**Models** (each with and without the explicit ICULOS input, each with a balanced variant):

1. **Our Transformer**: v3 architecture and hyperparameters, softmax attention.
2. **KA-Transformer under our protocol**: the paper's architecture (8 heads × 64 = 512, 4 layers, FFN 2048, dropout 0.2, RBF-kernel attention with learned β), adapted to a causal mask, per-hour last-position readout, our labels and horizons, the frozen split, forward-fill only. Trained with our recipe (optimiser, schedule, loss, batch size), so the comparison isolates the architecture.
3. **LightGBM** reference, plus a time-blind variant.

**Three things to say out loud when presenting this** (they are repeated next to the results):

- *The Transformer always knows the ICU hour.* It enters through the positional encoding of the absolute hour, through the time-since-measurement of features not yet measured, and through the number of visible past hours. So "±ICULOS" tests only **one extra explicit input**, not "with vs without time".
- *Hours 1–6 of early-onset stays are recognition, not prediction.* 65 % of septic stays have their onset before hour 12. The by-hour figures therefore come in three versions, and the early-onset number is never presented alone as early-prediction evidence.
- *VAL was used to tune our Transformer* (v3's search), but not KA or LightGBM. Comparisons on VAL favour our model somewhat. The one unbiased table is the frozen-model evaluation on EXT at the very end of the project.

The KA paper's own numbers (AUROC 0.96 / 0.94 / 0.98 at 1 / 6 / 12 h) are **not comparable** with anything below: it makes one prediction per patient from a window that ends before diagnosis, with ICU length of stay as an input. They are context only.

In [ ]:
# ============================ CELL 1 · BOOTSTRAP: clone the repo, import sepsis_core from it ============================
REPO_URL = "https://github.com/Myakey/sepsis-transformer-analysis.git"
REPO_REF = "main"                                  # branch, tag or commit hash. PIN A COMMIT HASH for any run you will report.
REPO_DIR = "/content/sepsis-transformer-analysis"

import os, sys, subprocess
sys.dont_write_bytecode = True
if "google.colab" in sys.modules and not os.path.exists("/content/drive/MyDrive"):      # Drive holds the data and the checkpoints
    from google.colab import drive
    drive.mount("/content/drive")

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f"git {' '.join(args[:2])} failed: {r.stderr.strip()[-300:]}")
    return r.stdout.strip()

def _auth_url(url):
    tok = None                                     # private repo: Colab secret GITHUB_TOKEN (Notebook access on)
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        tok = os.environ.get("GITHUB_TOKEN")
    return url.replace("https://", f"https://{tok}@") if tok else url

if not os.path.isdir(os.path.join(REPO_DIR, ".git")): _git("clone", "--quiet", _auth_url(REPO_URL), REPO_DIR)
else: _git("fetch", "--quiet", "--all", cwd=REPO_DIR)
_git("checkout", "--quiet", REPO_REF, cwd=REPO_DIR)
if REPO_REF in _git("branch", "--format=%(refname:short)", cwd=REPO_DIR).split(): _git("pull", "--quiet", "--ff-only", cwd=REPO_DIR)
_git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)        # never leave a token in the clone's config
CORE_COMMIT = _git("rev-parse", "HEAD", cwd=REPO_DIR)
os.environ["SEPSIS_PROJECT_ROOT"] = REPO_DIR
os.environ["SEPSIS_CORE_COMMIT"] = CORE_COMMIT
for m in [k for k in sys.modules if k == "sepsis_core" or k.startswith("sepsis_core.")]: del sys.modules[m]
if os.path.join(REPO_DIR, "Notebooks", "v4") not in sys.path: sys.path.insert(0, os.path.join(REPO_DIR, "Notebooks", "v4"))

import importlib.util
_need = [p for p in ("mlflow", "lightgbm") if importlib.util.find_spec(p) is None]
if _need: subprocess.call([sys.executable, "-m", "pip", "install", "-q", *_need])

import json, time, gc
import numpy as np, pandas as pd, torch
import sepsis_core
from sepsis_core import config as C, data, dataset, models, engine, metrics, calibration, balancing, controls, gbdt, runs, plots, tracking
print("sepsis_core", sepsis_core.__version__, "@ commit", CORE_COMMIT[:12], f"({REPO_REF})", "| torch", torch.__version__)

In [ ]:
# ============================ CELL 2 · SWITCHES ============================
SMOKE_TEST   = False
ICULOS_OPTS  = (False, True)       # explicit ICULOS input off / on
BALANCED     = True                # also train the balanced variant of every model
SMOTE_RATIO  = 0.25                # LightGBM only: synthetic positives up to this fraction of the negatives (1.0 = fully balanced; needs far more RAM)
KA_USE_MASK  = True                # KA also sees the measured/not-measured mask, as our model does (the paper is silent on this)
RUN_GBDT, RUN_TRANSFORMER, RUN_KA = True, True, True

runs.setup(smoke=SMOKE_TEST)
SEED = C.CFG.SEEDS[0]
HP = dict(C.V3_BEST_HP)
REC, GB, SCORES, REP = {}, {}, {}, {}
def mname(base, iculos, bal): return f"{base}{'_iculos' if iculos else ''}{'_bal' if bal else ''}_s{SEED}"

In [ ]:
# ============================ CELL 3 · DATA ============================
B = runs.prepare_mimic()
lens = B["store"].lens
print(f"stay length (h, after dropping post-onset rows): median {int(np.median(lens))}, p95 {int(np.percentile(lens, 95))}, max {int(lens.max())} "
      f"-> MAX_SEQ_LEN {C.CFG.MAX_SEQ_LEN}: one window per stay, every hour predicted once with its full history")

In [ ]:
# ============================ CELL 4 · LightGBM: no ICULOS column / + ICULOS / time-blind / SMOTE variants ============================
if RUN_GBDT:
    variants = {"gbdt": dict(), "gbdt_iculos": dict(with_iculos=True), "gbdt_timeblind": dict(time_blind=True)}
    if BALANCED: variants.update({"gbdt_smote": dict(smote_ratio=SMOTE_RATIO), "gbdt_iculos_smote": dict(with_iculos=True, smote_ratio=SMOTE_RATIO)})
    for nm, kw in variants.items():
        GB[nm] = gbdt.fit_or_load(runs.model_dir(nm), B["store"], B["pre"], B["fit_idx"], B["val_idx"], **kw)
        SCORES[nm] = (GB[nm]["rows"], gbdt.to_logit(GB[nm]["scores"]))
        m_ = metrics.compute_metrics(B["store"], *SCORES[nm])
        print(f"  {nm:<20s} VAL AUROC {m_['auroc_mean']:.4f} | trees {GB[nm]['best_iter']} | synthetic rows {GB[nm]['n_synthetic']} | top: {GB[nm]['top_features'][:5]}")
    print("note: 'gbdt' has no ICULOS column but still reads the hour through *_hrs_since of never-measured features; 'gbdt_timeblind' removes that channel.")

In [ ]:
# ============================ CELL 5 · OUR TRANSFORMER and KA-TRANSFORMER, +/- explicit ICULOS, +/- balancing ============================
for bal in ([False, True] if BALANCED else [False]):
    wrap = balancing.oversample_positive_windows(B["store"]) if bal else None
    for ic in ICULOS_OPTS:
        if RUN_TRANSFORMER:
            nm = mname("softmax", ic, bal)
            REC[nm] = runs.train_model(B, nm, {**HP, "arm": "softmax", "use_iculos": ic}, seed=SEED, role=("lecturer_balanced" if bal else "primary_arm" if not ic else "lecturer_iculos"), wrapper=wrap)
        if RUN_KA:
            nm = mname("ka", ic, bal)
            REC[nm] = runs.train_model(B, nm, {**HP, "arm": "ka", "use_iculos": ic, "aux_w": 0.0, "ka_use_mask": KA_USE_MASK}, seed=SEED,
                                       role=("lecturer_ka_balanced" if bal else "lecturer_ka"), wrapper=wrap)
for nm in REC: SCORES[nm] = runs.val_scores(nm)

## Results on VAL

Each model is calibrated on natural-distribution VAL (cross-fitted by stay) before thresholds and calibration are reported; balanced models need this because balancing distorts the base rate. Intervals are stay-level bootstrap CIs. This cell takes a couple of minutes per model.

In [ ]:
# ============================ CELL 6 · EVALUATE EVERY MODEL ============================
_rows0 = next(iter(SCORES.values()))[0]
for nm, (rows, lg) in SCORES.items():
    assert np.array_equal(rows, _rows0), f"{nm}: row order differs"
    if nm not in REP: REP[nm] = runs.evaluate(B, rows, lg)
    print(runs.line(nm, REP[nm]))

In [ ]:
# ============================ CELL 7 · PAIRED COMPARISONS (same VAL rows, stay-level paired bootstrap) ============================
def cmp(a, b, label):
    if a in SCORES and b in SCORES:
        r = runs.paired(B["store"], _rows0, SCORES[a][1], SCORES[b][1]); COMPARE[label] = dict(a=a, b=b, **r)
        print(f"  {label:<62s} {r['d']:+.4f} [{r['lo']:+.4f}, {r['hi']:+.4f}]  {'(b better)' if r['lo'] > 0 else '(a better)' if r['hi'] < 0 else '(no clear difference)'}")
COMPARE = {}
s0, k0 = mname("softmax", False, False), mname("ka", False, False)
print("Architecture (b - a):")
cmp(k0, s0, "our Transformer - KA-Transformer"); cmp("gbdt_iculos", s0, "our Transformer - LightGBM(+ICULOS)"); cmp("gbdt_iculos", k0, "KA-Transformer - LightGBM(+ICULOS)")
print("Explicit ICULOS input (b = with, a = without). The Transformers know the hour either way:")
cmp(s0, mname("softmax", True, False), "our Transformer: + explicit ICULOS"); cmp(k0, mname("ka", True, False), "KA-Transformer: + explicit ICULOS (the paper's LOS input)")
cmp("gbdt", "gbdt_iculos", "LightGBM: + ICULOS column"); cmp("gbdt_timeblind", "gbdt", "LightGBM: hour readable via hrs_since vs time-blind")
print("Balancing (b = balanced, a = natural distribution):")
for ic in ICULOS_OPTS:
    cmp(mname("softmax", ic, False), mname("softmax", ic, True), f"our Transformer{' +ICULOS' if ic else ''}: oversampled positive windows")
    cmp(mname("ka", ic, False), mname("ka", ic, True), f"KA-Transformer{' +ICULOS' if ic else ''}: oversampled positive windows")
cmp("gbdt", "gbdt_smote", "LightGBM: SMOTE"); cmp("gbdt_iculos", "gbdt_iculos_smote", "LightGBM +ICULOS: SMOTE")

In [ ]:
# ============================ CELL 8 · THE TABLE THE PAPER REPORTS: sensitivity / specificity / F1 / accuracy ============================
print("Threshold = F1-optimal, chosen on one half of VAL and scored on the other (cross-fitted), on calibrated probabilities.")
print("Accuracy is shown only because the KA paper reports it: at ~1% prevalence 'always negative' already gets the accuracy in the last column.\n")
print(f"{'model':<28s}{'h':>4s}{'sens':>8s}{'spec':>8s}{'prec':>8s}{'F1':>8s}{'acc':>8s}{'acc(always neg)':>17s}{'cal slope':>11s}{'ECE':>9s}")
for nm, rep in REP.items():
    for t, c in zip(rep["thresholds"], rep["calibration"]):
        print(f"{nm:<28s}{'t+' + str(t['horizon']):>4s}{t['sensitivity']:>8.3f}{t['specificity']:>8.3f}{t['precision']:>8.3f}{t['f1']:>8.3f}{t['accuracy']:>8.3f}{t['accuracy_always_negative']:>17.3f}{c['cal_slope']:>11.2f}{c['ece']:>9.4f}")

## How early can it tell?

Three views per figure: all stays; without early-onset stays (onset < 12 h); early-onset vs late-onset septic stays as separate curves (each against all non-septic stays). The third panel is the **recognition vs prediction** analysis: in early-onset stays, hours 1–6 mostly recognise sepsis that is already under way or imminent; late-onset stays show prediction.

In [ ]:
# ============================ CELL 9 · AUROC / AUPRC BY ICU HOUR ============================
main = {lab: REP[nm]["by_icu_hour"] for lab, nm in (("our Transformer", s0), ("KA-Transformer", k0), ("LightGBM +ICULOS", "gbdt_iculos")) if nm in REP}
if main:
    for h in (6, 1):
        plots.by_icu_hour(main, horizon=h, metric="auroc", path=runs.figure_path(f"01_auroc_by_hour_t{h}.png"))
    plots.by_icu_hour(main, horizon=6, metric="auprc", path=runs.figure_path("01_auprc_by_hour_t6.png"))
if s0 in REP:
    print("Recognition vs prediction, our Transformer, label at t+6 h (AUROC | positive rate | rows):")
    tab = REP[s0]["by_icu_hour"]
    for bucket in dict.fromkeys(r["bucket"] for r in tab):
        cells = {r["view"]: r for r in tab if r["bucket"] == bucket}
        print(f"  hours {bucket:<8s} all {cells['all']['auroc_t6']:.3f} | no early-onset {cells['no_early_onset']['auroc_t6']:.3f} | "
              f"early-onset (recognition) {cells['early_onset']['auroc_t6']:.3f} [{cells['early_onset']['pos_rate_t6']:.3%}, n={cells['early_onset']['n_rows']:,}] | "
              f"late-onset (prediction) {cells['late_onset']['auroc_t6']:.3f} [{cells['late_onset']['pos_rate_t6']:.3%}, n={cells['late_onset']['n_rows']:,}]")

In [ ]:
# ============================ CELL 10 · SAVE (this is what to send back) ============================
slim = {nm: {k: v for k, v in rep.items() if k not in ("calibrator",)} for nm, rep in REP.items()}
SUMMARY = dict(time=time.strftime("%Y-%m-%dT%H:%M:%S"), core_commit=CORE_COMMIT, smoke=SMOKE_TEST, seed=SEED, data_hash=B["data_hash"], cohort=B["stats"],
               smote_ratio=SMOTE_RATIO, ka_use_mask=KA_USE_MASK, reports=slim, comparisons=COMPARE,
               training={nm: dict(epochs_run=r["epochs_run"], soup_epochs=r["soup_epochs"], n_params=r["n_params"], epoch_sec=r["epoch_sec"]) for nm, r in REC.items()},
               gbdt={nm: dict(best_iter=g["best_iter"], n_synthetic=g["n_synthetic"], top_features=g["top_features"]) for nm, g in GB.items()},
               ka_paper_context="AUROC 0.962/0.944/0.984 at 1/6/12 h (Zhu et al. 2025). DIFFERENT TASK, NOT COMPARABLE: one prediction per patient, LOS as input.")
runs.save_json(SUMMARY, "01_summary.json")

## What to send back

The printed output of Cells 4, 6, 7, 8 and 9, or `reports/01_summary.json`. All aggregate.

**Balancing, in one paragraph for the lecturer.** The class imbalance (about 1 % positive hours) is handled in the main models by the loss (focal loss), not by resampling. For the balanced variants requested here, the sequence models repeat whole *real* training windows that contain a positive hour, on FIT only. SMOTE is not applied to sequences: it interpolates between two feature vectors, and an hourly ICU sequence is thousands of coupled values (measurements, their measured/not-measured flags, time since last measurement); interpolating two stays mixes hours that are not aligned and produces fractional "was measured" flags, so the synthetic stays are not plausible patients. SMOTE proper is used where it is valid: on LightGBM's per-hour feature rows, again on FIT only. Every balanced model is recalibrated on natural-distribution VAL, because balancing inflates predicted risk.

Not included: the PhysioNet-2019 utility score (the brief said "if it's cheap"; it needs its own tested implementation, so I left it out and will add it on request).